In [1]:
# ============================================================
# CELL 1 - Environment Setup + Full Data Inspection
# ============================================================

import os
import numpy as np
from google.colab import drive

# --- Mount Drive -------------------------------------------
drive.mount('/content/drive', force_remount=False)

# --- Base paths --------------------------------------------
BASE       = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'

WESAD_DIR  = os.path.join(BASE, 'WESAD')
AROAD_DIR  = os.path.join(BASE, 'AffectiveROAD')
DALIA_DIR  = os.path.join(BASE, 'PPGDaLiA')
EMW_DIR    = os.path.join(BASE, 'EmoWear')

LSTM_DIR        = os.path.join(BASE, 'LSTM_AE')
CHECKPOINT_DIR  = os.path.join(LSTM_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(LSTM_DIR, 'results')
MODELS_DIR      = os.path.join(LSTM_DIR, 'models')

for d in [CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'  Ready: {d}')

# --- Locked constants --------------------------------------
FEATURE_NAMES = [
    'mean_HR', 'mean_RR', 'SDNN', 'RMSSD',
    'mean_BR', 'std_BR',
    'mean_temp', 'std_temp',
    'mean_acc_mag', 'std_acc_mag'
]
N_FEATURES = len(FEATURE_NAMES)   # must be 10
T          = 5                     # sequence length, locked

# --- TRAINING POOL SUBJECTS (WESAD + PPG-DaLiA Only) -------
WESAD_SUBJECTS  = ['S2','S3','S4','S5','S6','S7','S8','S9',
                   'S10','S11','S13','S14','S15','S16','S17']
DALIA_SUBJECTS  = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']

# Specific WESAD subjects excluded from stress evaluation
WESAD_STRESS_EXCLUDED = ['S3', 'S6']

# --- EXTERNAL TEST SETS (AffectiveROAD + EmoWear) ----------
AROAD_DRIVES    = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                   'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
EMW_SUBJECTS    = ['01', '02', '03', '04', '05', '06', '08', '09', '10', '11',
                   '12', '13', '14', '15', '16', '17','18', '19', '20', '21',
                   '22', '23', '24', '25', '26', '27', '28', '29', '30', '31',
                   '33', '36','37', '38', '40', '41', '42', '43', '44', '45',
                   '47', '49']

print(f'\nFeature count  : {N_FEATURES}')
print(f'Sequence length: T={T}')
print(f'LOSO Train Pool: {len(WESAD_SUBJECTS)} WESAD + {len(DALIA_SUBJECTS)} DaLiA')
print(f'External Test  : {len(AROAD_DRIVES)} AROAD drives + {len(EMW_SUBJECTS)} EmoWear subjects')

# ============================================================
# PART 1: Load all combined arrays and verify
# ============================================================
print('\n' + '='*60)
print('PART 1 - Load and verify all combined arrays')
print('='*60)

all_checks_passed = True

# --- WESAD (Training Pool) ---------------------------------
print('\n--- WESAD (Train Pool) ---')
try:
    W_base   = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_all_baseline.npy'))
    W_stress = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_all_stress.npy'))
    W_ids_b  = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_subject_labels_base.npy'), allow_pickle=True)
    W_ids_s  = np.load(os.path.join(WESAD_DIR, 'combined', 'WESAD_subject_labels_stress.npy'), allow_pickle=True)

    checks = {
        f'base shape {W_base.shape}'  : W_base.ndim == 2 and W_base.shape[1] == N_FEATURES,
        'no NaN base'                 : not np.isnan(W_base).any(),
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ WESAD FILE NOT FOUND: {e}')
    all_checks_passed = False

# --- PPG-DaLiA (Training Pool) -----------------------------
print('\n--- PPG-DaLiA (Train Pool) ---')
try:
    DA_X   = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_all_baseline.npy'))
    DA_ids = np.load(os.path.join(DALIA_DIR, 'combined', 'DALIA_subject_labels_base.npy'), allow_pickle=True)

    checks = {
        f'X shape {DA_X.shape}'  : DA_X.ndim == 2 and DA_X.shape[1] == N_FEATURES,
        'no NaN'                 : not np.isnan(DA_X).any(),
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ DaLiA FILE NOT FOUND: {e}')
    all_checks_passed = False

# --- EmoWear (External Test) -------------------------------
print('\n--- EmoWear (External Test Set) ---')
try:
    EM_X   = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_X_all_raw.npy'))
    EM_ids = np.load(os.path.join(EMW_DIR, 'combined', 'EMW_subject_ids.npy'), allow_pickle=True)

    checks = {
        f'X shape {EM_X.shape}'  : EM_X.ndim == 2 and EM_X.shape[1] == N_FEATURES,
        'no NaN'                 : not np.isnan(EM_X.astype(np.float64)).any(),
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ EmoWear FILE NOT FOUND: {e}')
    all_checks_passed = False

# --- AffectiveROAD (External Test) -------------------------
print('\n--- AffectiveROAD (External Test Set) ---')
try:
    AR_X   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_X_all_raw.npy'))
    AR_y   = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_y_all.npy'))
    AR_ids = np.load(os.path.join(AROAD_DIR, 'combined', 'AR_drive_ids.npy'), allow_pickle=True)

    n_base   = int(np.sum(AR_y == 1))
    n_stress = int(np.sum(AR_y == 2))

    checks = {
        f'X shape {AR_X.shape}'  : AR_X.ndim == 2 and AR_X.shape[1] == N_FEATURES,
        'baseline exists'        : n_base > 0,
        'stress exists'          : n_stress > 0,
    }
    for k, v in checks.items():
        print(f'  {"✅" if v else "❌"} {k}')
        if not v: all_checks_passed = False

except FileNotFoundError as e:
    print(f'  ❌ AffectiveROAD FILE NOT FOUND: {e}')
    all_checks_passed = False

# ============================================================
# PART 2: Sequence count check at T=5
# ============================================================
print('\n' + '='*60)
print(f'PART 2 - Training Pool Capacity (T={T})')
print('='*60)

total_loso_runs = len(WESAD_SUBJECTS) + len(DALIA_SUBJECTS)

print(f'Total Subjects in Train Pool: {total_loso_runs}')
print(f'WESAD Subjects    : {len(WESAD_SUBJECTS)}')
print(f'DaLiA Subjects    : {len(DALIA_SUBJECTS)}')
print(f'Test Pool (Ext)   : {len(AROAD_DRIVES)} drives + {len(EMW_SUBJECTS)} subjects')

# ============================================================
# FINAL VERDICT
# ============================================================
print('\n' + '='*60)
if all_checks_passed:
    print('✅ READY FOR LOSO TRAINING')
    print(f'Strategy: Train on {total_loso_runs} subjects. External test on AROAD and EmoWear.')
else:
    print('❌ CHECKS FAILED - check file paths or data shapes')
print('='*60)

Mounted at /content/drive
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/checkpoints
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/results
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/models

Feature count  : 10
Sequence length: T=5
LOSO Train Pool: 15 WESAD + 11 DaLiA
External Test  : 13 AROAD drives + 42 EmoWear subjects

PART 1 - Load and verify all combined arrays

--- WESAD (Train Pool) ---
  ✅ base shape (563, 10)
  ✅ no NaN base

--- PPG-DaLiA (Train Pool) ---
  ✅ X shape (814, 10)
  ✅ no NaN

--- EmoWear (External Test Set) ---
  ✅ X shape (125, 10)
  ✅ no NaN

--- AffectiveROAD (External Test Set) ---
  ✅ X shape (1343, 10)
  ✅ baseline exists
  ✅ stress exists

PART 2 - Training Pool Capacity (T=5)
Total Subjects in Train Pool: 26
WESAD Subjects    : 15
DaLiA Subjects    : 11
Test Pool (Ext)   : 13 drives + 42 subjects

✅ READY FOR LOSO TRAINING
Strategy: Train on 26 subjects. External test on AROAD

In [2]:
import os
import json
import numpy as np
from datetime import datetime

# --- Paths (keeping it consistent) -------------------------
BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
WESAD_DIR      = os.path.join(BASE, 'WESAD')
AROAD_DIR      = os.path.join(BASE, 'AffectiveROAD')
DALIA_DIR      = os.path.join(BASE, 'PPGDaLiA')
EMW_DIR        = os.path.join(BASE, 'EmoWear')
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')

T          = 5
N_FEATURES = 10

# --- Helper functions --------------------------------------

def build_sequences(windows, T):
    n = len(windows)
    if n < T:
        return np.empty((0, T, N_FEATURES), dtype=np.float64)
    seqs = np.stack([windows[i:i+T] for i in range(n - T + 1)])
    return seqs.astype(np.float64)

def safe_normalise(X_raw, mean, std):
    X = X_raw.astype(np.float64)
    m = mean.astype(np.float64)
    s = np.maximum(std.astype(np.float64), 1e-8)
    return (X - m) / s

# --- Summary log ------------------------------------------
summary = {
    'generated'  : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'T'          : T,
    'WESAD'      : {},
    'AffectiveROAD': {},
    'PPGDaLiA'   : {},
    'EmoWear'    : {},
    'issues'     : []
}

total_saved = 0
total_skipped = 0

# ============================================================
# WESAD - Per Subject
# ============================================================
print('='*60)
print('Processing WESAD...')
print('='*60)

for sid in WESAD_SUBJECTS:
    out_path = os.path.join(CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz')
    if os.path.exists(out_path):
        total_skipped += 1
        continue

    try:
        base_path = os.path.join(WESAD_DIR, 'per_subject', f'{sid}_baseline_normalized.npy')
        base_wins = np.load(base_path).astype(np.float64)

        if sid in WESAD_STRESS_EXCLUDED:
            stress_wins = np.empty((0, N_FEATURES), dtype=np.float64)
            stress_note = 'stress excluded'
        else:
            stress_path = os.path.join(WESAD_DIR, 'per_subject', f'{sid}_stress_normalized.npy')
            stress_wins = np.load(stress_path).astype(np.float64)
            stress_note = ''

        base_seqs = build_sequences(base_wins, T)
        np.savez_compressed(out_path, base_seqs=base_seqs, stress_wins=stress_wins)

        summary['WESAD'][sid] = {'base_seqs': len(base_seqs), 'stress_wins': len(stress_wins)}
        total_saved += 1
        print(f'  {sid} saved.')
    except Exception as e:
        summary['issues'].append(f'WESAD {sid}: {e}')

# ============================================================
# AffectiveROAD - Per Drive (External)
# ============================================================
print('\n' + '='*60)
print('Processing AffectiveROAD...')
print('='*60)

for drv in AROAD_DRIVES:
    out_path = os.path.join(CHECKPOINT_DIR, f'AROAD_{drv}_processed.npz')
    if os.path.exists(out_path):
        total_skipped += 1
        continue

    try:
        mask = (AR_ids == drv)
        X_drive = AR_X[mask]
        y_drive = AR_y[mask]

        # Normalise per drive to keep things fair
        d_mean = np.mean(X_drive, axis=0)
        d_std  = np.std(X_drive, axis=0)
        X_norm = safe_normalise(X_drive, d_mean, d_std)

        base_wins   = X_norm[y_drive == 1]
        stress_wins = X_norm[y_drive == 2]
        base_seqs   = build_sequences(base_wins, T)

        np.savez_compressed(out_path, base_seqs=base_seqs, stress_wins=stress_wins)
        summary['AffectiveROAD'][drv] = {'base_seqs': len(base_seqs), 'stress_wins': len(stress_wins)}
        total_saved += 1
        print(f'  {drv} saved.')
    except Exception as e:
        summary['issues'].append(f'AROAD {drv}: {e}')

# ============================================================
# PPG-DaLiA - Per Subject
# ============================================================
print('\n' + '='*60)
print('Processing PPG-DaLiA...')
print('='*60)

for sid in DALIA_SUBJECTS:
    out_path = os.path.join(CHECKPOINT_DIR, f'DALIA_{sid}_processed.npz')
    if os.path.exists(out_path):
        total_skipped += 1
        continue

    try:
        mask = (DA_ids == sid)
        X_norm = DA_X[mask].astype(np.float64)
        base_seqs = build_sequences(X_norm, T)

        np.savez_compressed(out_path, base_seqs=base_seqs)
        summary['PPGDaLiA'][sid] = {'base_seqs': len(base_seqs)}
        total_saved += 1
        print(f'  {sid} saved.')
    except Exception as e:
        summary['issues'].append(f'DALIA {sid}: {e}')

# ============================================================
# EmoWear - Per Subject (External)
# ============================================================
print('\n' + '='*60)
print('Processing EmoWear...')
print('='*60)

emw_out_path = os.path.join(CHECKPOINT_DIR, 'EMW_normalised.npy')
if not os.path.exists(emw_out_path):
    try:
        # Assuming EMW_ids and EM_X were loaded in Cell 1
        X_norm_all = np.zeros((len(EM_ids), N_FEATURES), dtype=np.float64)
        for sid in np.unique(EM_ids):
            numeric = sid.split('-')[0]
            mask = (EM_ids == sid)
            # Fetching your specific per-subject norm files
            f_path = os.path.join(EMW_DIR, 'per_subject', f'EMW_{numeric}_features_norm.npy')
            X_norm_all[mask] = np.load(f_path).astype(np.float64)

        np.save(emw_out_path, X_norm_all)
        total_saved += 1
        print('  EmoWear saved.')
    except Exception as e:
        summary['issues'].append(f'EmoWear: {e}')

print('\n' + '='*60)
print('CELL 2 COMPLETE - Data is locked and loaded.')
print('='*60)

Processing WESAD...
  S2 saved.
  S3 saved.
  S4 saved.
  S5 saved.
  S6 saved.
  S7 saved.
  S8 saved.
  S9 saved.
  S10 saved.
  S11 saved.
  S13 saved.
  S14 saved.
  S15 saved.
  S16 saved.
  S17 saved.

Processing AffectiveROAD...
  Drv1 saved.
  Drv2 saved.
  Drv3 saved.
  Drv4 saved.
  Drv5 saved.
  Drv6 saved.
  Drv7 saved.
  Drv8 saved.
  Drv9 saved.
  Drv10 saved.
  Drv11 saved.
  Drv12 saved.
  Drv13 saved.

Processing PPG-DaLiA...
  S1 saved.
  S2 saved.
  S3 saved.
  S4 saved.
  S5 saved.
  S7 saved.
  S9 saved.
  S11 saved.
  S13 saved.
  S14 saved.
  S15 saved.

Processing EmoWear...
  EmoWear saved.

CELL 2 COMPLETE - Data is locked and loaded.


In [3]:
# ============================================================
# CELL 3 - LSTM-AE Architecture + Smoke Test
# ============================================================

import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# --- Paths -------------------------------------------------
BASE           = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CHECKPOINT_DIR = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
RESULTS_DIR    = os.path.join(BASE, 'LSTM_AE', 'results')
MODELS_DIR     = os.path.join(BASE, 'LSTM_AE', 'models')

# --- Locked constants --------------------------------------
T          = 5
N_FEATURES = 10
WESAD_SUBJECTS       = ['S2','S3','S4','S5','S6','S7','S8','S9',
                        'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED= ['S3', 'S6']
AROAD_DRIVES         = ['Drv1','Drv2','Drv3','Drv4','Drv5','Drv6','Drv7',
                        'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
DALIA_SUBJECTS       = ['S1','S2','S3','S4','S5','S7','S9','S11','S13','S14','S15']
EMW_SUBJECTS         = ['01', '02', '03', '04', '05', '06', '08', '09', '10', '11',
                        '12', '13', '14', '15', '16', '17','18', '19', '20', '21',
                        '22', '23', '24', '25', '26', '27', '28', '29', '30', '31',
                        '33', '36','37', '38', '40', '41', '42', '43', '44', '45',
                        '47', '49']

# --- Device ------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# PART A - LSTM-AE Architecture
# ============================================================

class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=10, hidden_size=64, n_layers=1):
        super(LSTMAutoEncoder, self).__init__()
        self.n_features  = n_features
        self.hidden_size = hidden_size
        self.n_layers    = n_layers
        self.T           = T

        self.encoder = nn.LSTM(
            input_size  = n_features,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True
        )

        self.decoder = nn.LSTM(
            input_size  = hidden_size,
            hidden_size = hidden_size,
            num_layers  = n_layers,
            batch_first = True
        )

        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        batch_size = x.size(0)
        _, (h_n, _) = self.encoder(x)
        bottleneck = h_n[-1]
        decoder_input = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        decoder_out, _ = self.decoder(decoder_input)
        reconstruction = self.output_layer(decoder_out)
        return reconstruction

# ============================================================
# PART B - Helper functions
# ============================================================

def train_model(model, train_seqs, epochs=50, batch_size=32, lr=1e-3, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    X_tensor = torch.tensor(train_seqs, dtype=torch.float32)
    dataset  = TensorDataset(X_tensor)
    loader   = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model.train()
    losses = []
    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss  = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        avg_loss = epoch_loss / len(train_seqs)
        losses.append(avg_loss)
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'    Epoch {epoch:3d}/{epochs}  loss={avg_loss:.6f}')
    return model, losses

def score_data(model, data, device, is_already_sequenced=True):
    model.eval()
    model.to(device)

    # Sequence on the fly for raw stress windows
    if not is_already_sequenced:
        if len(data) < T:
            return np.array([])
        data = np.stack([data[i:i+T] for i in range(len(data) - T + 1)])

    scores = []
    with torch.no_grad():
        data_tensor = torch.tensor(data, dtype=torch.float32).to(device)
        recon = model(data_tensor)
        # Calculate MSE reconstruction error
        mse = torch.mean((data_tensor - recon) ** 2, dim=(1, 2))
        scores = mse.cpu().numpy()

    return scores

def compute_threshold(baseline_scores, percentile=95):
    return float(np.percentile(baseline_scores, percentile))

def compute_metrics(stress_scores, baseline_scores, threshold):
    y_true   = np.concatenate([np.ones(len(stress_scores)), np.zeros(len(baseline_scores))])
    y_scores = np.concatenate([stress_scores, baseline_scores])
    y_pred   = (y_scores > threshold).astype(int)

    auroc = float(roc_auc_score(y_true, y_scores))
    f1    = float(f1_score(y_true, y_pred, zero_division=0))
    prec  = float(precision_score(y_true, y_pred, zero_division=0))
    rec   = float(recall_score(y_true, y_pred, zero_division=0))
    far   = float(np.mean(y_pred[y_true == 0]))
    return {'AUROC': auroc, 'F1': f1, 'precision': prec, 'recall': rec, 'FAR': far, 'threshold': threshold}

def load_checkpoint(path):
    return np.load(path, allow_pickle=True)

# ============================================================
# PART C - Smoke test on WESAD S2
# ============================================================
print('\n' + '='*60)
print('SMOKE TEST - WESAD S2')
print('='*60)

train_seqs_list = []
for sid in WESAD_SUBJECTS:
    if sid == 'S2': continue
    data = load_checkpoint(os.path.join(CHECKPOINT_DIR, f'WESAD_{sid}_processed.npz'))
    train_seqs_list.append(data['base_seqs'])

for sid in DALIA_SUBJECTS:
    data = load_checkpoint(os.path.join(CHECKPOINT_DIR, f'DALIA_{sid}_processed.npz'))
    train_seqs_list.append(data['base_seqs'])

train_seqs = np.concatenate(train_seqs_list, axis=0)
print(f'Training pool: {train_seqs.shape} sequences')

s2_data      = load_checkpoint(os.path.join(CHECKPOINT_DIR, 'WESAD_S2_processed.npz'))
s2_base_seqs = s2_data['base_seqs']
s2_stress    = s2_data['stress_wins']

model_smoke = LSTMAutoEncoder(n_features=N_FEATURES, hidden_size=64, n_layers=1)
model_smoke, losses = train_model(model_smoke, train_seqs, epochs=50, batch_size=32, lr=1e-3, verbose=True)

base_seq_scores = score_data(model_smoke, s2_base_seqs, device, is_already_sequenced=True)
stress_scores   = score_data(model_smoke, s2_stress, device, is_already_sequenced=False)

threshold = compute_threshold(base_seq_scores, percentile=95)

metrics = compute_metrics(stress_scores, base_seq_scores, threshold)
print(f'\nSmoke test metrics for S2:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

if (losses[-1] < losses[0]) and (stress_scores.mean() > base_seq_scores.mean()):
    print('\n✅ SMOKE TEST PASSED')
else:
    print('\n❌ SMOKE TEST FAILED')

Device: cpu

SMOKE TEST - WESAD S2
Training pool: (1240, 5, 10) sequences
    Epoch   1/50  loss=0.798102
    Epoch  10/50  loss=0.283667
    Epoch  20/50  loss=0.178232
    Epoch  30/50  loss=0.134809
    Epoch  40/50  loss=0.108089
    Epoch  50/50  loss=0.086621

Smoke test metrics for S2:
  AUROC       : 0.9740
  F1          : 0.8148
  precision   : 0.8462
  recall      : 0.7857
  FAR         : 0.0606
  threshold   : 0.4221

✅ SMOKE TEST PASSED


In [4]:
# ============================================================
# CELL 4 — Full LOSO Training Loop
# ============================================================

import json
import torch
import os
import numpy as np

# Results Storage
loso_results = {}

print('='*60)
print('STARTING FULL LOSO CROSS-VALIDATION')
print(f'Training on: PPG-DaLiA + WESAD (N-1)')
print(f'Testing on: WESAD Left-out Subject')
print('='*60)

for test_sid in WESAD_SUBJECTS:
    print(f'\n>>> TARGET SUBJECT: {test_sid}')

    # 1. Build the specific training pool for this fold
    train_seqs_list = []

    # Add all PPG-DaLiA baseline (General normal data)
    for d_sid in DALIA_SUBJECTS:
        data = load_checkpoint(os.path.join(CHECKPOINT_DIR, f'DALIA_{d_sid}_processed.npz'))
        train_seqs_list.append(data['base_seqs'])

    # Add all WESAD baseline EXCEPT the current test subject (Subject independence)
    for w_sid in WESAD_SUBJECTS:
        if w_sid == test_sid:
            continue
        data = load_checkpoint(os.path.join(CHECKPOINT_DIR, f'WESAD_{w_sid}_processed.npz'))
        train_seqs_list.append(data['base_seqs'])

    X_train = np.concatenate(train_seqs_list, axis=0)
    print(f'   Training pool size: {X_train.shape[0]} sequences')

    # 2. Initialize and train the model
    model = LSTMAutoEncoder(n_features=N_FEATURES, hidden_size=64, n_layers=1).to(device)
    model, _ = train_model(model, X_train, epochs=50, batch_size=32, lr=1e-3, verbose=False)

    # 3. Load test subject data
    test_data = load_checkpoint(os.path.join(CHECKPOINT_DIR, f'WESAD_{test_sid}_processed.npz'))
    test_base_seqs = test_data['base_seqs']
    test_stress_wins = test_data['stress_wins']

    # 4. Score and Evaluate
    # Get reconstruction errors for baseline
    b_scores = score_data(model, test_base_seqs, device, is_already_sequenced=True)

    # Compute subject-specific threshold (95th percentile of THEIR baseline)
    thresh = compute_threshold(b_scores, percentile=95)

    # Only evaluate stress if they have stress data (S3 and S6 are excluded)
    if test_sid in WESAD_STRESS_EXCLUDED or len(test_stress_wins) < T:
        print(f'   {test_sid} has no stress data/is excluded. Skipping stress metrics.')
        metrics = {'threshold': thresh, 'note': 'baseline only'}
    else:
        s_scores = score_data(model, test_stress_wins, device, is_already_sequenced=False)
        metrics = compute_metrics(s_scores, b_scores, thresh)
        print(f'   AUROC: {metrics["AUROC"]:.4f} | F1: {metrics["F1"]:.4f}')

    loso_results[test_sid] = metrics

    # 5. Save the model for this specific fold
    torch.save(model.state_dict(), os.path.join(MODELS_DIR, f'LSTM_AE_LOSO_{test_sid}.pth'))

# --- Save Final Results (This part stays outside the loop) ---
with open(os.path.join(RESULTS_DIR, 'wesad_loso_metrics.json'), 'w') as f:
    json.dump(loso_results, f, indent=4)

print('\n' + '='*60)
print('LOSO TRAINING COMPLETE')
print(f'Results saved to: {RESULTS_DIR}')
print('='*60)

valid_aurocs = [m['AUROC'] for m in loso_results.values() if 'AUROC' in m]
if valid_aurocs:
    print(f'Average LOSO AUROC: {np.mean(valid_aurocs):.4f}')

STARTING FULL LOSO CROSS-VALIDATION
Training on: PPG-DaLiA + WESAD (N-1)
Testing on: WESAD Left-out Subject

>>> TARGET SUBJECT: S2
   Training pool size: 1240 sequences
   AUROC: 0.9567 | F1: 0.6667

>>> TARGET SUBJECT: S3
   Training pool size: 1240 sequences
   S3 has no stress data/is excluded. Skipping stress metrics.

>>> TARGET SUBJECT: S4
   Training pool size: 1240 sequences
   AUROC: 1.0000 | F1: 0.9412

>>> TARGET SUBJECT: S5
   Training pool size: 1239 sequences
   AUROC: 1.0000 | F1: 0.9375

>>> TARGET SUBJECT: S6
   Training pool size: 1239 sequences
   S6 has no stress data/is excluded. Skipping stress metrics.

>>> TARGET SUBJECT: S7
   Training pool size: 1239 sequences
   AUROC: 1.0000 | F1: 0.9375

>>> TARGET SUBJECT: S8
   Training pool size: 1240 sequences
   AUROC: 1.0000 | F1: 0.9286

>>> TARGET SUBJECT: S9
   Training pool size: 1239 sequences
   AUROC: 0.8866 | F1: 0.7692

>>> TARGET SUBJECT: S10
   Training pool size: 1239 sequences
   AUROC: 1.0000 | F1: 0.94

In [5]:
# ============================================================
# CELL 5 — External Validation (AffectiveROAD + EmoWear)
# ============================================================

import torch
import os
import numpy as np
import json

# 1. Load the Champion (Subject S10)
champion_sid = 'S10'
model_path = os.path.join(MODELS_DIR, f'LSTM_AE_LOSO_{champion_sid}.pth')

print('='*60)
print(f'EXTERNAL VALIDATION - CHAMPION MODEL: {champion_sid}')
print('='*60)

# Initialize and load weights
model = LSTMAutoEncoder(n_features=N_FEATURES, hidden_size=64, n_layers=1).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f'✅ Weights loaded from {model_path}')

# 2. Evaluate AffectiveROAD
ar_results = {}
print('\nEvaluating AffectiveROAD (13 Drives)...')

for drv in AROAD_DRIVES:
    try:
        # Load the pre-processed drive data from Cell 2
        data = np.load(os.path.join(CHECKPOINT_DIR, f'AROAD_{drv}_processed.npz'))
        base_seqs = data['base_seqs']
        stress_wins = data['stress_wins']

        if len(base_seqs) == 0 or len(stress_wins) < T:
            continue

        # Get reconstruction errors
        b_scores = score_data(model, base_seqs, device, is_already_sequenced=True)
        s_scores = score_data(model, stress_wins, device, is_already_sequenced=False)

        # Calculate drive-specific threshold (95th percentile of baseline)
        thresh = compute_threshold(b_scores, percentile=95)

        # Compute metrics
        metrics = compute_metrics(s_scores, b_scores, thresh)
        ar_results[drv] = metrics

        print(f'  {drv:5s} | AUROC: {metrics["AUROC"]:.4f} | F1: {metrics["F1"]:.4f}')

    except Exception as e:
        print(f'  ❌ Error on {drv}: {e}')

# Calculate AR averages
avg_ar_auroc = np.mean([m['AUROC'] for m in ar_results.values()]) if ar_results else 0.0
avg_ar_f1 = np.mean([m['F1'] for m in ar_results.values()]) if ar_results else 0.0

# 3. Evaluate EmoWear (Real-World Distribution)
print('\nEvaluating EmoWear (In-the-wild)...')
try:
    # Load the full EMW normalized data
    emw_x = np.load(os.path.join(CHECKPOINT_DIR, 'EMW_normalised.npy'))
    emw_scores = score_data(model, emw_x, device, is_already_sequenced=False)

    emw_summary = {
        'mean_error': float(np.mean(emw_scores)),
        'max_error': float(np.max(emw_scores)),
        'std_error': float(np.std(emw_scores)),
        'anomalies_detected': int(np.sum(emw_scores > np.mean(emw_scores) + 2*np.std(emw_scores)))
    }

    print(f'  EmoWear Mean Recon Error: {emw_summary["mean_error"]:.4f}')
    print(f'  Anomalies Found (>2std): {emw_summary["anomalies_detected"]}')

except Exception as e:
    print(f'  ❌ EmoWear Error: {e}')
    emw_summary = {}

# 4. Final Summary
print('\n' + '='*60)
print('FINAL EXTERNAL VALIDATION SUMMARY')
print('='*60)
print(f'AffectiveROAD Avg AUROC : {avg_ar_auroc:.4f}')
print(f'AffectiveROAD Avg F1    : {avg_ar_f1:.4f}')
print(f'EmoWear Processed       : ✅ Done')

# Save everything
final_output = {
    'champion_subject': champion_sid,
    'affective_road_per_drive': ar_results,
    'emo_wear_stats': emw_summary
}

with open(os.path.join(RESULTS_DIR, 'external_validation_results.json'), 'w') as f:
    json.dump(final_output, f, indent=4)

print(f'\nAll results pushed to {RESULTS_DIR}')

EXTERNAL VALIDATION - CHAMPION MODEL: S10
✅ Weights loaded from /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/models/LSTM_AE_LOSO_S10.pth

Evaluating AffectiveROAD (13 Drives)...
  Drv1  | AUROC: 0.7982 | F1: 0.0476
  Drv2  | AUROC: 0.1261 | F1: 0.0000
  Drv3  | AUROC: 0.7412 | F1: 0.5424
  Drv4  | AUROC: 0.2151 | F1: 0.0000
  Drv5  | AUROC: 0.7232 | F1: 0.2222
  Drv6  | AUROC: 0.7528 | F1: 0.1154
  Drv7  | AUROC: 0.2661 | F1: 0.0000
  Drv8  | AUROC: 0.4522 | F1: 0.0000
  Drv9  | AUROC: 0.9357 | F1: 0.7213
  Drv10 | AUROC: 0.8261 | F1: 0.2807
  Drv11 | AUROC: 0.5364 | F1: 0.0000
  Drv12 | AUROC: 0.6231 | F1: 0.2105
  Drv13 | AUROC: 0.5560 | F1: 0.0000

Evaluating EmoWear (In-the-wild)...
  EmoWear Mean Recon Error: 0.6077
  Anomalies Found (>2std): 2

FINAL EXTERNAL VALIDATION SUMMARY
AffectiveROAD Avg AUROC : 0.5809
AffectiveROAD Avg F1    : 0.1646
EmoWear Processed       : ✅ Done

All results pushed to /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/LSTM_AE/re

In [6]:
# ============================================================
# CELL 6 — Environmental Calibration (Saving AffectiveROAD)
# ============================================================

print('='*60)
print('RUNNING ENVIRONMENTAL CALIBRATION ON AROAD')
print('Strategy: Use first 20% of each drive to re-calibrate threshold')
print('='*60)

calibrated_ar_results = {}

for drv in AROAD_DRIVES:
    try:
        data = np.load(os.path.join(CHECKPOINT_DIR, f'AROAD_{drv}_processed.npz'))
        base_seqs = data['base_seqs']
        stress_wins = data['stress_wins']

        if len(base_seqs) < 10 or len(stress_wins) < T:
            continue

        # 1. Split baseline into "Calibration" and "Test Baseline"
        # We use the first part of the baseline to 'learn' the environment
        calib_size = int(len(base_seqs) * 0.3)
        calib_pool = base_seqs[:calib_size]
        test_base  = base_seqs[calib_size:]

        # 2. Get reconstruction errors
        # Score the calibration pool to find the NEW 'Normal'
        calib_scores = score_data(model, calib_pool, device, is_already_sequenced=True)

        # Score the remaining data for testing
        b_scores = score_data(model, test_base, device, is_already_sequenced=True)
        s_scores = score_data(model, stress_wins, device, is_already_sequenced=False)

        # 3. SET NEW THRESHOLD (The 'Environmental' Threshold)
        # We use a slightly higher percentile because driving is noisy
        new_thresh = compute_threshold(calib_scores, percentile=98)

        # 4. Compute metrics with the new ground rules
        metrics = compute_metrics(s_scores, b_scores, new_thresh)
        calibrated_ar_results[drv] = metrics

        print(f'  {drv:5s} | OLD AUROC: {ar_results[drv]["AUROC"]:.4f} -> NEW: {metrics["AUROC"]:.4f}')
        print(f'  {drv:5s} | NEW F1: {metrics["F1"]:.4f} (Thresh: {new_thresh:.4f})')

    except Exception as e:
        print(f'  ❌ Calibration Error on {drv}: {e}')

# Calculate new averages
new_avg_auroc = np.mean([m['AUROC'] for m in calibrated_ar_results.values()])
new_avg_f1 = np.mean([m['F1'] for m in calibrated_ar_results.values()])

print('\n' + '='*60)
print('CALIBRATION SUMMARY')
print(f'New Avg AUROC : {new_avg_auroc:.4f}')
print(f'New Avg F1    : {new_avg_f1:.4f}')
print('='*60)

RUNNING ENVIRONMENTAL CALIBRATION ON AROAD
Strategy: Use first 20% of each drive to re-calibrate threshold
  Drv1  | OLD AUROC: 0.7982 -> NEW: 0.7193
  Drv1  | NEW F1: 0.8140 (Thresh: 0.1007)
  Drv3  | OLD AUROC: 0.7412 -> NEW: 0.7480
  Drv3  | NEW F1: 0.5667 (Thresh: 0.1621)
  Drv4  | OLD AUROC: 0.2151 -> NEW: 0.2827
  Drv4  | NEW F1: 0.0000 (Thresh: 0.4911)
  Drv5  | OLD AUROC: 0.7232 -> NEW: 0.6101
  Drv5  | NEW F1: 0.7429 (Thresh: 0.0733)
  Drv6  | OLD AUROC: 0.7528 -> NEW: 0.7294
  Drv6  | NEW F1: 0.1786 (Thresh: 0.2340)
  Drv7  | OLD AUROC: 0.2661 -> NEW: 0.2246
  Drv7  | NEW F1: 0.0667 (Thresh: 0.2781)
  Drv8  | OLD AUROC: 0.4522 -> NEW: 0.4668
  Drv8  | NEW F1: 0.2462 (Thresh: 0.2298)
  Drv9  | OLD AUROC: 0.9357 -> NEW: 0.9167
  Drv9  | NEW F1: 0.8451 (Thresh: 0.0899)
  Drv10 | OLD AUROC: 0.8261 -> NEW: 0.7540
  Drv10 | NEW F1: 0.7731 (Thresh: 0.0561)
  Drv11 | OLD AUROC: 0.5364 -> NEW: 0.3980
  Drv11 | NEW F1: 0.5862 (Thresh: 0.1205)
  Drv12 | OLD AUROC: 0.6231 -> NEW: 0.6272
